# ChromaDB basics: write, browse, and why the embedder matters

Env: `conda activate PyTorchTutorial` (chromadb 1.5.9).

| notebook | what it covers |
| --- | --- |
| `why_rag.ipynb` | why stuffing the prompt stops working |
| `rag_with_a_database.ipynb` | embeddings + Chroma + RAG loop (800 Helios docs) |
| **this one** | write / browse / filters / the embedder + ONNX MiniLM in depth |
| `index.ipynb` + `scripts/mini_rag.py` | chunking, hybrid search, reranking |

Run top to bottom. The DB is recreated each run so output is deterministic.

## 0. Big picture first: tokenizer is NOT the embedding

Three different mappings — do not conflate them:

| # | name | input → output | example | purpose |
| --- | --- | --- | --- | --- |
| 1 | **Tokenizer** | text → list of ints (token IDs) | `"hello world"` → `[7592, 2088]` on MiniLM (different ints on Qwen/Google) | turn strings into integers a model can index; reversible via decode |
| 2 | **LLM token vectors** (inside the generator) | each ID → one vector (e.g. 4096-d), transformer mixes them | N tokens → N vectors → next-token prediction | **generation**; lives inside qwen/llama/gemma, never stored in Chroma |
| 3 | **Sentence vector** (retriever — this notebook) | whole text → **one** vector (MiniLM 384-d, bge-m3 1024-d, OpenAI 1536-d) | whole paragraph → 1 × 384 numbers | **retrieval**; cosine = meaning similarity; stored in Chroma |

### 0.0 The simple version (read this first)

**LLM:** sentence → tokenizer → IDs (ints, model-specific) → project each ID into a space like 4096-d → forward through the network → output logits → turn into the next token. The vectors are scratch paper: used once per forward pass, then gone.

**Chroma:** sentence → embedder → **one** 384-d vector for the whole sentence. That vector is never turned back into tokens — it is only coordinates for *meaning*. Chroma stores it **next to** the original text:

```
doc-1 "Eiffel Tower is in Paris" → [0.02, -0.11, ...] (384 numbers) + the text
doc-2 "On-call owned by Mirembe"  → [0.05,  0.03, ...] (384 numbers) + the text
```

A query takes the same trip through the *same* embedder, then Chroma dots it against every stored doc (vectors are L2-normalised, so dot = cosine):

```
query "who handles night emergencies?" → [0.04, 0.02, ...]
  query · doc-1 = 0.15
  query · doc-2 = 0.37  ← winner → Chroma returns doc-2's TEXT
```

The winning **text** goes into the LLM prompt. So yes: **Chroma only returns sentences and knows nothing about your LLM** — any generator (qwen/llama/gemma/gpt) can consume them. The 384 numbers did their only job: ranking. That is also why docs and query must share the embedder (same coordinates), while the LLM is freely swappable.

### 0.1 One sentence, three tokenizers — same text, different IDs

`"The Eiffel Tower is in Paris."` tokenizes differently on every model family, because vocabularies differ. The live cell below shows MiniLM WordPiece (30,522 entries, verified from its `vocab.txt`): ~10 IDs. The same string on Qwen BPE (~152k vocab) or Gemma SentencePiece (~256k vocab) yields *different IDs and a different count*. IDs are model-local — never compare or reuse them across models.

### 0.2 What an LLM does during training

1. **Lookup:** each ID selects one row of a learned table of shape `vocab × hidden`. E.g. Llama-3.1-8B: 128,256 × 4096 — about half a billion parameters just for this table.
2. **Contextualize:** transformer blocks mix those N vectors with attention (position-aware).
3. **Predict + learn:** a head maps the last vector to logits over the whole vocab; cross-entropy loss against the true next token backpropagates into *everything*, including the lookup table.

So LLM token vectors are trained for one skill: **predicting what comes next**. Reference sizes (per model card / `config.json` — re-check before citing):

| model | vocab | hidden = token-vector dim | layers |
| --- | --- | --- | --- |
| MiniLM-L6-v2 (retriever backbone, for scale) | 30,522 | 384 | 6 |
| Llama-3.1-8B | 128,256 | 4096 | 32 |
| Qwen2.5-7B | ~152,064 | 3584 | 28 |
| Gemma-2-9B (Google) | 256,128 | 3584 | 42 |

### 0.3 What an LLM does in a forward pass (inference)

Identical path minus the loss: tokenize prompt → lookup → transformer → logits → sample **one** token → append it → repeat (autoregressive; KV-cache only skips recompute). At no point does a single sentence vector appear — the N vectors stay N vectors until the final sampling step.

### 0.4 What the sentence embedder does instead (§5)

Same start (MiniLM: tokenize → 30,522 × 384 lookup → 6 transformer layers), then a different ending: **mean-pool** over the real (non-pad) token vectors → **L2-normalize** → exactly **one** 384-d vector. Trained with contrastive/ranking loss (pull paraphrases together, push unrelated texts apart), never next-token loss. Cosine between two such vectors ≈ meaning similarity — the only property Chroma relies on.

**Do LLM dims and sentence-vector dims need to match? No.** The 4096-d vectors live and die inside one LLM forward pass; the 384-d vectors live in Chroma. The LLM only ever receives retrieved *text*, never the 384-d numbers. The only pair that must share dim + model is doc-vector vs query-vector (both type 3, same embedder) — that is §5c.

### 0.5 Then Chroma, end to end

```
corpus → chunk (≤256 tokens, §5c) → sentence-embed in ONE space → Chroma
      (N x D float32 + HNSW graph + text/metadata in sqlite)
query  → embed with the SAME model → nearest neighbours → retrieved TEXT
      → prompt → ANY LLM (qwen/llama/gemma/gpt, §5d) → answer
```

**One collection = one sentence-embedder (§5–5c). Tokenizers and LLM internals can all differ (§5d).**

### 0.6 Storing sentences: the knobs and their impact (detail: §5e)

Per doc Chroma stores the vector (HNSW `*.bin`) plus id + text + metadata (`chroma.sqlite3`). Your choices:

| # | knob | impact in one line |
| --- | --- | --- |
| 1 | **Embedding model** (384 vs 1024 vs 1536) | biggest quality lever; changing it invalidates all stored vectors → new collection + re-add |
| 2 | **Distance space** (`cosine`/`l2`/`ip`, fixed at creation) | must match your vectors; wrong space = silently wrong ranking |
| 3 | **HNSW params** (`M`/`construction_ef`/`search_ef` = 16/100/10) | recall vs RAM/build-time/query-time; matters only at scale |
| 4 | **`add` vs `upsert` + id scheme** | dedup behaviour: `add` rejects dup ids, `upsert` overwrites, hash ids make re-indexing a no-op |
| 5 | **Metadata** (`where=` filters) | exact-match pre-filter: fast and precise, never semantic |
| 6 | **Auto vs manual embeddings** | manual gives control but you own dims + normalisation |
| 7 | **Persistent vs Ephemeral client** | disk (survives restarts, skips re-embedding) vs RAM-only scratchpad |

Rule of thumb: pick 1 once per collection, leave 2–3 on defaults until thousands of docs, use 4–5 for hygiene. Next cell runs 1 vs 3 live on MiniLM.

In [ ]:
# 1 vs 2 vs 3, live on MiniLM + documented LLM shapes for contrast
from chromadb.utils.embedding_functions import ONNXMiniLM_L6_V2

_demo_ef = ONNXMiniLM_L6_V2()
tok = _demo_ef.tokenizer.encode("The Eiffel Tower is in Paris.")
real_ids = [i for i in tok.ids if i != 0]  # 0 = [PAD], padded to 256 inside Chroma's tokenizer
vec = _demo_ef(["The Eiffel Tower is in Paris."])[0]
vocab = sum(1 for _ in open(_demo_ef.DOWNLOAD_PATH / "onnx" / "vocab.txt"))

print(f"1. tokenizer (MiniLM WordPiece, vocab={vocab}): '{'The Eiffel Tower is in Paris.'}'")
print(f"   -> {len(real_ids)} IDs: {real_ids} (+{len(tok.ids)-len(real_ids)} PADs to 256)")
print("   Qwen/Gemma give DIFFERENT IDs for this string (own vocabs ~152k/256k).")
print(f"2. LLM lookup (NOT run, shapes only): Llama-8B table 128256x4096, Qwen7B ~152kX3584, Gemma-9B 256128x3584")
print(f"   this sentence -> N IDs -> N x hidden vectors -> next-token logits over vocab")
print("3. embedder (live): text ->", vec.shape, "ONE vector (mean-pool + L2-norm of the N token vectors)")
print("   decode back:", _demo_ef.tokenizer.decode(real_ids))
print("\nChroma stores only type 3. The LLM in §5d receives the decoded TEXT, never this vector.")

In [ ]:
import shutil
import sqlite3
from pathlib import Path

import numpy as np
import chromadb
from chromadb.utils.embedding_functions import ONNXMiniLM_L6_V2

# Resolve a persist dir that works whether Jupyter was started at repo root or in rag/
cwd = Path.cwd()
if cwd.name == "rag":
    PERSIST = str(cwd / "chroma_basics_db")
elif (cwd / "rag").exists():
    PERSIST = str(cwd / "rag" / "chroma_basics_db")
else:
    PERSIST = str(cwd / "chroma_basics_db")

shutil.rmtree(PERSIST, ignore_errors=True)
Path(PERSIST).mkdir(parents=True, exist_ok=True)
print("persist:", PERSIST)
print("chromadb:", chromadb.__version__)

## 1. Connect + create collection

`PersistentClient(path=...)` opens (or creates) the on-disk DB. `get_or_create_collection` is a table with one distance metric. We pass the embedder explicitly so §5 can inspect it — omitting it uses the same default.

In [ ]:
client = chromadb.PersistentClient(path=PERSIST)
ef = ONNXMiniLM_L6_V2()  # THE default: all-MiniLM-L6-v2 over ONNX, see §5
col = client.get_or_create_collection(
    "basics", embedding_function=ef, metadata={"hnsw:space": "cosine"}
)
print(f"collection '{col.name}' count={col.count()}")
print("embedder:", type(col._embedding_function).__name__)

## 2. Write: add / upsert

`add` inserts. `upsert` inserts-or-replaces by id. Because we gave no `embeddings=`, Chroma calls `ef(documents)` for us.

In [ ]:
ids = ["doc-1", "doc-2", "doc-3", "doc-4"]
docs = [
    "ChromaDB stores vectors plus the original text and metadata.",
    "PyTorch tensors run on GPU for fast neural network training.",
    "The Eiffel Tower is in Paris and was built in 1889.",
    "On-call for the data platform is owned by Mirembe Okonkwo.",
]
metas = [
    {"topic": "db", "source": "tutorial"},
    {"topic": "ml", "source": "tutorial"},
    {"topic": "geo", "source": "tutorial"},
    {"topic": "ops", "source": "handbook"},
]
col.add(ids=ids, documents=docs, metadatas=metas)
print("count after add:", col.count())

## 3. Browse without searching: peek / get

`peek` = quick sample. `get` = exact lookup by id or metadata filter. No vectors involved.

In [ ]:
p = col.peek(2)
for i, d in zip(p["ids"], p["documents"]):
    print(f"  {i}: {d[:60]}...")

all_data = col.get(include=["documents", "metadatas"])
for i, d, m in zip(all_data["ids"], all_data["documents"], all_data["metadatas"]):
    print(f"  {i} {m} -> {d[:50]}...")

In [ ]:
# WHERE-style filters + id lookup
print("topic=ml:", col.get(where={"topic": "ml"}, include=["documents"]) ["ids"])
print("by id:", col.get(ids=["doc-3"], include=["documents", "metadatas"]))

## 4. Query: semantic search

`query_texts` are embedded with the *same* `ef`, then HNSW finds nearest neighbours. `distances` are cosine distances; similarity = `1 - dist`.

In [ ]:
q = col.query(
    query_texts=["who handles night emergencies for data platform?"],
    n_results=2, include=["documents", "metadatas", "distances"],
)
for cid, doc, meta, dist in zip(q["ids"][0], q["documents"][0], q["metadatas"][0], q["distances"][0]):
    print(f"  sim={1-dist:.3f} {cid} {meta} -> {doc[:60]}...")

print("\nwith where filter:")
qf = col.query(query_texts=["tall tower in France"], n_results=2, where={"source": "tutorial"})
for cid, doc, dist in zip(qf["ids"][0], qf["documents"][0], qf["distances"][0]):
    print(f"  {cid} dist={dist:.3f} -> {doc[:60]}...")

## 5. Embedder in depth: what it is and what ONNX MiniLM is

An **embedding function** has one job: `vectors = ef(list_of_strings)`. Chroma calls it on `add` (docs) and on `query` (query_texts) so both land in the **same vector space**. Everything else — HNSW index, filters, persistence — is just storage around those vectors.

**The bundled ONNX model is exactly one model**, hardcoded in `chromadb/utils/embedding_functions/onnx_mini_lm_l6_v2.py`:

- `MODEL_NAME = all-MiniLM-L6-v2` — 6-layer MiniLM, the sentence-transformers classic
- 384 dimensions, `max_tokens = 256`, L2-normalised, `default_space = cosine`
- Downloaded once from `https://chroma-onnx-models.s3.amazonaws.com/all-MiniLM-L6-v2/onnx.tar.gz` to `~/.cache/chroma/onnx_models/all-MiniLM-L6-v2/` (model.onnx + tokenizer.json + vocab.txt), SHA-checked
- Runs on **onnxruntime + tokenizers**, not torch — that is the whole point: zero torch dependency, CPU-friendly, reproducible. `preferred_providers` (CPU/CUDA/CoreML) only changes *speed*, never the vectors
- `DefaultEmbeddingFunction` is an alias for this class. There is no `all-MiniLM-L12-v2-ONNX` or `paraphrase-ONNX` bundled

Consequences: docs longer than 256 tokens raise `ValueError` (you must chunk first — see `scripts/mini_rag.py:chunk_text` and `index.ipynb`), and swapping to any other model changes every vector (see §5c).

In [ ]:
# Inspect the function behind the collection
print("name:", ef.name(), "| default_space:", ef.default_space(), "| spaces:", ef.supported_spaces())
print("max_tokens:", ef.max_tokens(), "| model:", ef.MODEL_NAME)
print("url:", ef.MODEL_DOWNLOAD_URL)
print("cache:", ef.DOWNLOAD_PATH)
print("cached files:", sorted(p.name for p in (ef.DOWNLOAD_PATH / "onnx").glob("*")))

v = np.asarray(ef(["hello world", "hello world"]))
print("\nshape:", v.shape, "(2 docs x 384 dims)")
print("norms:", np.linalg.norm(v, axis=1).round(4), "<- L2-normalised, so dot == cosine")
print("self-similarity:", float(v[0] @ v[1]).__round__(4), "(identical texts -> ~1.0)")

### 5b. Concrete vendors: model, dims, and what Chroma stores

Same interface (`ef(texts) -> vectors`), different numbers. Dims are per model card — always re-check, vendors version them.

| vendor | Chroma class | model | dims | runs where |
| --- | --- | --- | --- | --- |
| Chroma default (**this notebook**, verified below) | `ONNXMiniLM_L6_V2` | all-MiniLM-L6-v2 | **384**, 256 tok/doc, cosine | local CPU (onnxruntime, `~/.cache/chroma/onnx_models/`) |
| open-source HF | `SentenceTransformerEmbeddingFunction` | `all-MiniLM-L12-v2` / `all-mpnet-base-v2` / `BAAI/bge-m3` | **384 / 768 / 1024** | local torch + HF download; bge-m3 is multilingual, long-context |
| Qwen (local server) | `OllamaEmbeddingFunction` | `qwen3-embedding:0.6b` — the one in `rag_with_a_database.ipynb`, which printed `array(10, 1024)` | **1024** | `ollama serve` + `ollama pull qwen3-embedding:0.6b` |
| OpenAI (cloud) | `OpenAIEmbeddingFunction` | `text-embedding-3-small` / `-large` | **1536 / 3072** (Matryoshka: truncatable, e.g. 512) | API key + network |
| Google (cloud) | `GoogleGenerativeAIEmbeddingFunction` | `text-embedding-004` / `gemini-embedding` | **768** / up to **3072** via `output_dimensionality` | API key + network |
| manual | `embeddings=` | anything (incl. TF-IDF fallback in `mini_rag.py`) | must equal the collection's D | you batch and normalise |

**What Chroma does with D:** each collection fixes D at first `add` and stores an N×D float32 matrix + HNSW graph (`*.bin`) + text/metadata (`chroma.sqlite3`). A 384-d collection and a 1536-d collection coexist as *separate collections* — but one collection never mixes dims, and a query must carry exactly D numbers (demo next cell). Compare with LLM internals: Llama-3.1-8B token vectors are 4096-d (hidden size, vocab ~128k) — a different model, a different D, never stored here.

Switching embedder = new collection + re-`add` everything. Swapping the *generator* LLM needs nothing (see §5d).

In [ ]:
# Same interface, different Ds. Only the default runs offline; the rest are shown
# as documented dims (no server/key needed) + a live per-collection-D demo.
print(f"this notebook (ONNX MiniLM, live): 1 doc -> {np.asarray(ef(['probe'])).shape} (expect (1, 384))")
print("qwen3-embedding:0.6b (rag_with_a_database.ipynb, live there): (N, 1024)")
print("OpenAI text-embedding-3-small / -large (model card): 1536 / 3072")
print("Google text-embedding-004 / gemini-embedding (model card): 768 / up to 3072")
print("BAAI/bge-m3 (model card): 1024")

try:
    from chromadb.utils.embedding_functions import OllamaEmbeddingFunction
    oef = OllamaEmbeddingFunction(url="http://localhost:11434/api/embed", model_name="qwen3-embedding:0.6b")
    print("\nollama live check: qwen dims =", len(oef(["probe"])[0]))
except Exception as e:
    print(f"\nollama not running ({type(e).__name__}) — documented 1024 stands, collection above unaffected")

# What Chroma enforces: D is per collection. Fake a 1536-d (OpenAI-sized)
# collection with explicit embeddings — no API key needed for the demo.
rng = np.random.RandomState(1)
fake1536 = rng.randn(2, 1536).astype(np.float32)
fake1536 /= np.linalg.norm(fake1536, axis=1, keepdims=True)
big = client.get_or_create_collection("demo_1536")
try:
    big.add(ids=["o1", "o2"], documents=["openai-sized doc A", "openai-sized doc B"], embeddings=fake1536.tolist())
except Exception as e:
    print("add 1536-d:", type(e).__name__, str(e)[:100])
print(f"collection 'basics' holds 384-d x{col.count()}, collection 'demo_1536' holds 1536-d x{big.count()} — side by side, fine")
try:
    big.query(query_embeddings=[([0.0] * 384)], n_results=1)  # 384-d query on 1536-d collection
except Exception as e:
    print(f"384-d query on 1536-d collection -> {type(e).__name__}: {str(e)[:110]}...")
client.delete_collection("demo_1536")  # keep the DB clean for §6

### 5c. The one rule: docs and query must share the space

A distance is only meaningful between vectors from the **same model + same preprocessing**. Two failure modes:

1. **Different dims → loud error.** A 1024-d query against this 384-d collection is rejected.
2. **Same dims, different model → silent garbage.** Two 384-d models (L6 vs L12) both return numbers, but nearest neighbours are meaningless. Nothing warns you — you must track which embedder a collection uses.

Related limit: the ONNX model caps at 256 tokens per document. Longer text is rejected (not truncated), which is why real pipelines chunk first.

In [ ]:
# 1. Wrong dims -> Chroma refuses (good)
try:
    col.query(query_embeddings=[[0.0] * 1024], n_results=1)
except Exception as e:
    print(f"1024-d query on 384-d collection -> {type(e).__name__}: {str(e)[:120]}...")

# 2. Same dims, different vectors: distance is only comparable within one function
a = np.asarray(ef(["on-call rota owner"]))[0]
b = np.asarray(ef(["who is on call?"]))[0]
c = np.asarray(ef(["eiffel tower paris"]))[0]
print(f"\nsame-model sim(related)={(a @ b):.3f} vs sim(unrelated)={(a @ c):.3f}  <- signal")
rng = np.random.RandomState(0)
fake = rng.randn(384).astype(np.float32); fake /= np.linalg.norm(fake)
print(f"other-space sim={(a @ fake):.3f}  <- noise; this is what cross-model comparison looks like")

# 3. 256-token cap -> chunk instead of stuffing whole files
long_doc = "word " * 500
try:
    ef([long_doc])
except Exception as e:
    print(f"\n500-word doc -> {type(e).__name__}: {str(e)[:100]}...")
print("fix: scripts/mini_rag.py:chunk_text (400 words, 60 overlap) / index.ipynb §chunking")

### 5d. LLMs are not embedders: swap generators freely

Retrieval (embedder) and generation (LLM) touch different things: the embedder makes vectors, the LLM reads the retrieved **strings**. So one collection serves any LLM — `qwen3:30b`, `llama3`, GPT — with no re-embedding. Only changing the **embedder** forces a rebuild.

`rag_with_a_database.ipynb` §4 does exactly this: `retrieve()` (vectors) → `context` (text) → `ollama.chat(model=CHAT_MODEL, ...)` (LLM). Change `CHAT_MODEL`, keep the collection.

In [ ]:
# Same retrieved text feeds any LLM — no vectors leave the collection
hits = col.query(query_texts=["how do I get a new API credential?"], n_results=1)["documents"][0]
context = "\n".join(f"- {d}" for d in hits)
for llm in ["qwen3:30b-a3b-instruct-2507-q4_K_M", "llama3.1:8b", "gpt-4o"]:
    prompt = f"[LLM={llm}] Answer using ONLY:\n{context}\nQ: how do I get a new API credential?"
    print(prompt[:110] + "...")
print("\ncollection untouched: count =", col.count())

### 5e. Storing sentences in Chroma: the knobs and their impact

Chroma stores per doc: the 384-d vector (HNSW `*.bin`), plus id + text + metadata (`chroma.sqlite3`). Each choice below changes what is stored or how it is searched:

| # | knob | what it is | impact |
| --- | --- | --- | --- |
| 1 | **Embedding model** (`embedding_function=`, §5b) | which space + D (384 vs 1024 vs 1536) | biggest quality lever: jargon/multilingual need bigger models; changing it invalidates every stored vector → new collection + re-`add` |
| 2 | **Distance space** (`metadata={"hnsw:space": ...}` at creation) | `cosine` (default, right for normalised text vectors), `l2`, or `ip` | must match your vectors: normalised → cosine and `l2` rank identically; unnormalised custom embeddings + wrong space = silently wrong ranking; fixed at creation |
| 3 | **HNSW parameters** (`hnsw:M`, `hnsw:construction_ef`, `hnsw:search_ef`; defaults 16 / 100 / 10) | graph width, build-time and query-time candidate lists | higher = better recall, slower build + more RAM (`M`, `construction_ef`) or slower queries (`search_ef`); only matters at scale, not at 4 docs |
| 4 | **`add` vs `upsert` + id scheme** | `add` rejects duplicate ids, `upsert` overwrites; `mini_rag.py` uses content-hash ids | ids decide dedup + incremental indexing: same text re-added is a no-op with hash ids, an error with `add`, an overwrite with `upsert` |
| 5 | **Metadata** (`metadatas=`, `where=` / `where_document=`) | exact-match key/values stored in sqlite alongside vectors | `where={"topic":"ml"}` pre-filters before vector search (fast, precise) but is literal, never semantic |
| 6 | **Embeddings auto vs manual** | omit `embeddings=` (ef embeds) vs pass your own | manual gives control (custom pooling, TF-IDF) but you own batching, dims and normalisation — mismatch breaks ranking |
| 7 | **Client type** (`PersistentClient` vs `EphemeralClient`) | on-disk dir vs in-RAM only | persistent survives restarts and skips re-embedding (§6); ephemeral is a scratchpad for tests |

Rule of thumb: pick knob 1 once per collection, leave 2–3 on defaults until you have thousands of docs, and use 4–5 for hygiene (dedup + filters).

In [ ]:
# Knobs 2–3 live on the collection; 5 is exercised in §3–4. Inspect them here:
print("collection metadata:", col.metadata)  # hnsw:space cosine, rest are HNSW defaults

# Why cosine is the default: our vectors are normalised, so cosine and l2 rank identically
q = np.asarray(ef(["who handles night emergencies for data platform?"]))[0]
docs_m = np.asarray(ef([d for d in docs]))
cos = docs_m @ q                      # cosine similarity (higher = better)
l2 = np.linalg.norm(docs_m - q, axis=1)  # euclidean distance (lower = better)
print("cos ranking:", np.argsort(-cos).tolist(), " l2 ranking:", np.argsort(l2).tolist(), " <- same order")

## 6. Update, persistence, and what is on disk

`upsert` edits one id without re-embedding the rest. The persist dir holds `chroma.sqlite3` (ids, text, metadata) plus one HNSW segment dir (`*.bin`). For custom embedders, reopen with the **same** function or `query_texts` will embed in the wrong space.

In [ ]:
col.upsert(ids=["doc-3"], documents=["The Eiffel Tower is in Paris, France."],
           metadatas=[{"topic": "geo", "source": "tutorial"}])
print("edited:", col.get(ids=["doc-3"])["documents"])

# Fresh client = what a second process sees. No re-adding.
col2 = chromadb.PersistentClient(path=PERSIST).get_collection("basics")
print(f"reopened count={col2.count()}, collections={[c.name for c in chromadb.PersistentClient(path=PERSIST).list_collections()]}")
# custom embedder reopen would be: get_collection("basics", embedding_function=my_ef)

print("\non disk:")
for p in sorted(Path(PERSIST).rglob("*")):
    print(" ", p.relative_to(PERSIST))

con = sqlite3.connect(str(Path(PERSIST) / "chroma.sqlite3"))
print("\nsqlite tables:", [r[0] for r in con.execute("SELECT name FROM sqlite_master WHERE type='table'")][:6], "...")
print("collections:", list(con.execute("SELECT name FROM collections")))
con.close()

### Cheat sheet

| task | call |
| --- | --- |
| create | `get_or_create_collection(name, embedding_function=ef, metadata={"hnsw:space":"cosine"})` |
| add | `add(ids=, documents=, metadatas=)` (or `embeddings=` to bypass `ef`) |
| look | `peek(n)`, `get(ids=)`, `get(where={"topic":"ml"})` |
| search | `query(query_texts=[...], n_results=k, where=...)` |
| edit | `upsert(ids=, documents=, metadatas=)` · delete with `delete(ids=)` |
| reset | `rm -rf <persist>` (`chroma*` is gitignored) |

Delete `./rag/chroma_basics_db` to start over.